In [4]:
import pandas as pd

nodes = pd.read_csv("../mc1_csv/mc1_nodes.csv")
edges = pd.read_csv("../mc1_csv/mc1_edges.csv")

# Split nodes
persons  = nodes[nodes["Node Type"] == "Person"].copy()
songs    = nodes[nodes["Node Type"] == "Song"].copy()
albums   = nodes[nodes["Node Type"] == "Album"].copy()
all_works = pd.concat([songs, albums])

# Split edges
performer_of = edges[edges["Edge Type"] == "PerformerOf"]
in_style_of  = edges[edges["Edge Type"] == "InStyleOf"]
member_of    = edges[edges["Edge Type"] == "MemberOf"]

# Fix dtypes
all_works["release_date"]   = pd.to_numeric(all_works["release_date"],   errors="coerce")
all_works["notoriety_date"] = pd.to_numeric(all_works["notoriety_date"], errors="coerce")

# Find Sailor Shift
SAILOR_ID = nodes[nodes["name"] == "Sailor Shift"]["id"].values[0]  # 17255

# Oceanus Folk work IDs
of_ids = set(all_works[all_works["genre"] == "Oceanus Folk"]["id"])

In [5]:
# ── Step 1: Map each artist to their works ───────────────────────────────────
artist_works = performer_of.merge(
    all_works[["id","genre","release_date","notable","notoriety_date"]],
    left_on="target", right_on="id", how="left"
).rename(columns={"source": "artist_id"})

# Map artist_id -> set of their work IDs (for influence lookups)
work_ids_by_artist = performer_of.groupby("source")["target"].apply(set).to_dict()

# ── Step 2: Career metrics per artist ────────────────────────────────────────
metrics = artist_works.groupby("artist_id").agg(
    debut_year      = ("release_date",    "min"),
    latest_year     = ("release_date",    "max"),
    total_works     = ("id",              "count"),
    notable_works   = ("notable",         "sum"),   # boolean sum = count of True
    first_notoriety = ("notoriety_date",  "min"),
    last_notoriety  = ("notoriety_date",  "max"),
    genre_diversity = ("genre",           "nunique"),
).reset_index()

metrics["notoriety_lag"] = metrics["first_notoriety"] - metrics["debut_year"]
metrics["career_span"]   = metrics["latest_year"] - metrics["debut_year"]

# ── Step 3: Influence metrics ─────────────────────────────────────────────────
metrics["outbound_influence"] = metrics["artist_id"].map(
    lambda aid: in_style_of[in_style_of["source"].isin(work_ids_by_artist.get(aid, set()))].shape[0]
)
metrics["inbound_influence"] = metrics["artist_id"].map(
    lambda aid: in_style_of[in_style_of["target"].isin(work_ids_by_artist.get(aid, set()))].shape[0]
)

# Oceanus Folk alignment — how much does this artist draw from OF
metrics["of_alignment"] = metrics["artist_id"].map(
    lambda aid: in_style_of[
        (in_style_of["source"].isin(work_ids_by_artist.get(aid, set()))) &
        (in_style_of["target"].isin(of_ids))
    ].shape[0]
)

# ── Step 4: Rising Star Score (normalised) ────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler

score_cols = ["outbound_influence", "inbound_influence", "notable_works", "genre_diversity"]
scaler = MinMaxScaler()
norm = scaler.fit_transform(metrics[score_cols].fillna(0))
metrics[["out_norm","in_norm","notable_norm","diversity_norm"]] = norm

metrics["rising_star_score"] = (
    metrics["out_norm"]       * 0.4 +
    metrics["in_norm"]        * 0.2 +
    metrics["notable_norm"]   * 0.2 +
    metrics["diversity_norm"] * 0.2
)

# Attach names
metrics = metrics.merge(
    persons[["id","name","stage_name"]], left_on="artist_id", right_on="id", how="left"
)

metrics.to_csv("artist_metrics.csv", index=False)